In [3]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def to_document(doc, dept, status, text):
    return Document(
        page_content=text,
        metadata={
            "doc": doc,
            "dept": dept,
            "status": status,
        }
    )

In [8]:
CORPUS = [
    to_document("DOC001", "HR", "Active", "Employee onboarding policy and benefits overview for 2026."),
    to_document("DOC002", "HR", "Draft", "Updated remote work guidelines and home office stipend procedures."),
    to_document("DOC003", "Finance", "Approved", "Q1 2026 financial summary report and budget allocations."),
    to_document("DOC004", "Finance", "Pending", "Vendor invoice processing steps and auditing guidelines."),
    to_document("DOC005", "Engineering", "Active", "System architecture overview and database microservice schema."),
    to_document("DOC006", "Engineering", "Draft", "API security guidelines and OAuth 2.0 authentication flow."),
    to_document("DOC007", "Legal", "Archived", "Standard master services agreement template from 2024."),
    to_document("DOC008", "Legal", "Approved", "Data privacy compliance framework following GDPR and CCPA standards."),
    to_document("DOC009", "Marketing", "Active", "Q3 product launch brand identity and social media campaign assets."),
    to_document("DOC010", "Marketing", "Pending", "Customer feedback survey analysis and market research findings."),
    to_document("DOC011", "Operations", "Active", "Supply chain logistics workflow and warehouse safety guidelines."),
    to_document("DOC012", "Operations", "Draft", "Disaster recovery plan and incident escalation matrix."),
    to_document("DOC013", "Sales", "Approved", "Enterprise sales strategy and competitive analysis matrix."),
    to_document("DOC014", "Support", "Active", "Level 1 customer support troubleshooting manual and FAQ list."),
]

print(len(CORPUS), "documents")

14 documents


# Creating Embedding(L2 Normalized) and Text Splitter(Chunking) Instance

In [5]:
embedding=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

splitter=RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

c:\Users\Shubham Singh\Developer\AIEngineering\pythonAIConcepts\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Shubham Singh\Developer\AIEngineering\pythonAIConcepts\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shubham Singh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either ne

# Creating a Lambda Function

In [9]:
CURRENT=lambda d: d.metadata.get("status")=="current"

def build_all(docs=None):
    global store, chunks
    chunks=splitter.split_documents(docs or CORPUS)
    store=InMemoryVectorStore(embedding=embedding)
    store.add_documents(chunks)

In [10]:
build_all()

print(len(chunks), "chunks")

14 chunks


### Sample Query to get Embedding of Input Text

In [11]:
vector=embedding.embed_query("What's the refund policy")
print(len(vector), vector[:6])

384 [-0.09544149041175842, 0.04163969308137894, 0.0530204251408577, 0.02047557756304741, 0.06370741128921509, 0.053197115659713745]


# Retrieve the Top K Chunks based on Similarity Score

In [13]:
def retrieve(user_query: str, k=3):
    return store.similarity_search_with_score(user_query, k)

def show(result):
    for doc, score in result:
        print(f"{score:.3f} [{doc.metadata['status']}] {doc.metadata['doc']}: {doc.page_content[:100]}....")

In [14]:
QUESTION="Find the onboarding policy and benefits overview."
show(retrieve(QUESTION))

0.754 [Active] DOC001: Employee onboarding policy and benefits overview for 2026.....
0.288 [Archived] DOC007: Standard master services agreement template from 2024.....
0.272 [Active] DOC014: Level 1 customer support troubleshooting manual and FAQ list.....
